### Softmax Regression on MNIST

#### Implemention of softmax regression (multinomial logistic regression)

In [14]:
import time
import torch
import torch.nn.functional as F
from torchvision import datasets
from torch.utils.data import DataLoader
from torchvision import transforms

In [15]:
# Hyperparameters
batch_size = 256
learning_rate = 0.1
seed = 123
num_epochs = 32

# Architecture
num_features = 784
num_classes = 10

In [16]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [17]:
train_dataset = datasets.MNIST(root='data',
               train=True,
               transform=transforms.ToTensor(),
               download=True)

test_dataset = datasets.MNIST(root='data',
               train=False,
               transform=transforms.ToTensor()
               )

train_loader = DataLoader(dataset=train_dataset,
                          batch_size=batch_size,
                          shuffle=True)

test_loader = DataLoader(dataset=test_dataset,
                         batch_size=batch_size,
                         shuffle=False)

In [18]:
for images, labels in train_loader:
    print(f"Image batch dimensions: {images.shape} #NCHW")
    print(f"(N)Number of training examples(aka; batch size): {images.size(0)}")
    print(f"(C)Number of channels: {images.size(1)}")
    print(f"(H)Height X (W)Width of an Image: {images.size(2)} X {images.size(3)}")
    print(f"Image label dimensions: {labels.shape}")
    break

Image batch dimensions: torch.Size([256, 1, 28, 28]) #NCHW
(N)Number of training examples(aka; batch size): 256
(C)Number of channels: 1
(H)Height X (W)Width of an Image: 28 X 28
Image label dimensions: torch.Size([256])


In [19]:
# There are images from 1 - 9 letters written.
torch.unique(labels)

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [20]:
class SoftmaxRegression(torch.nn.Module):

    def __init__(self, num_features, num_classes):
        super(SoftmaxRegression, self).__init__()
        self.linear = torch.nn.Linear(num_features, num_classes)

        self.linear.weight.detach().zero_()
        self.linear.bias.detach().zero_()

    def forward(self, x):
        logits = self.linear(x)
        probs = F.softmax(logits, dim=1)
        return logits, probs
    

model = SoftmaxRegression(num_features=num_features,
                          num_classes=num_classes)

model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [21]:
torch.manual_seed(seed)

def computer_accuarcy(model, data_loader):
    correct_pred, num_examples = 0, 0

    for features, labels in data_loader:
        features = features.view(-1, num_features).to(device)
        labels = labels.to(device)
        logits, probs = model(features)
        _, predicted_labels = torch.max(probs, 1)
        num_examples += labels.size(0)
        correct_pred += (predicted_labels == labels).sum()
    return correct_pred.float() / num_examples * 100

In [24]:
start_time = time.time()
epoch_cost = []
for epoch in range(num_epochs):
    avg_cost = 0.0
    for batch_idx, (features, labels) in enumerate(train_loader):

        features = features.view(-1, num_features).to(device)
        labels = labels.to(device)

        logits, probs = model(features)
        cost = F.cross_entropy(logits, labels)
        optimizer.zero_grad()
        cost.backward()
        avg_cost += cost

        optimizer.step()

        if not batch_idx % 50:
            print(f"Epoch: {epoch+1:03d}/{32:03d} | Batch {batch_idx:03d}/{len(train_loader)} | Cost: {cost:.4f}")

    with torch.set_grad_enabled(False):
        avg_cost = avg_cost / len(train_loader)
        epoch_cost.append(avg_cost)
        print(f"Epoch: {epoch+1:03d}/{32:03d} Train Cost: {avg_cost:.4f}")
        print(f"Time elapsed: {(time.time() - start_time)/60:.2f} min")


Epoch: 001/032 | Batch 000/235 | Cost: 0.4108
Epoch: 001/032 | Batch 050/235 | Cost: 0.3229
Epoch: 001/032 | Batch 100/235 | Cost: 0.3850
Epoch: 001/032 | Batch 150/235 | Cost: 0.4311
Epoch: 001/032 | Batch 200/235 | Cost: 0.4073
Epoch: 001/032 Train Cost: 0.3683
Time elapsed: 0.14 min
Epoch: 002/032 | Batch 000/235 | Cost: 0.3603
Epoch: 002/032 | Batch 050/235 | Cost: 0.4002
Epoch: 002/032 | Batch 100/235 | Cost: 0.3556
Epoch: 002/032 | Batch 150/235 | Cost: 0.3599
Epoch: 002/032 | Batch 200/235 | Cost: 0.2676
Epoch: 002/032 Train Cost: 0.3512
Time elapsed: 0.25 min
Epoch: 003/032 | Batch 000/235 | Cost: 0.3414
Epoch: 003/032 | Batch 050/235 | Cost: 0.3581
Epoch: 003/032 | Batch 100/235 | Cost: 0.2045
Epoch: 003/032 | Batch 150/235 | Cost: 0.3521
Epoch: 003/032 | Batch 200/235 | Cost: 0.3447
Epoch: 003/032 Train Cost: 0.3392
Time elapsed: 0.36 min
Epoch: 004/032 | Batch 000/235 | Cost: 0.3916
Epoch: 004/032 | Batch 050/235 | Cost: 0.3527
Epoch: 004/032 | Batch 100/235 | Cost: 0.3435
E

In [25]:
import plotly.express as px

fig = px.line(x=range(1, len(epoch_cost)+1), y=epoch_cost, title='Training Loss')
fig.update_xaxes(title_text='Epochs')
fig.update_yaxes(title_text='Cross Entropy')
fig.show()

In [26]:
print(f"Test accuracy: {computer_accuarcy(model, test_loader):.2f}%")

Test accuracy: 92.36%


In [27]:
for features, labels in test_loader:
    break

fig = px.imshow(features[0].view(28, 28).cpu().numpy(), color_continuous_scale='gray')
fig.show()

In [28]:
_, predictions = model.forward(features[0].view(-1, num_features).to(device))
predictions = torch.argmax(predictions, dim=1)
print(f"Model prediction: {predictions.item()}")

Model prediction: 7
